In [2]:
!pip install pyserial

In [72]:
import serial, time
!pip install pyserial

In [73]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [74]:
print(serial)

<module 'serial' from 'C:\\Users\\boome\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [75]:
print(serial.__file__)

C:\Users\boome\anaconda3\Lib\site-packages\serial\__init__.py


In [76]:
print(serial.__version__)

3.5


In [77]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [78]:
baudrate = 115200

In [79]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [80]:
#ser.close()

In [81]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [82]:
ser.in_waiting

0

In [83]:
#ser.close()

In [84]:
#read_all(ser)

In [85]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [86]:
read_all(ser)

''

In [87]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [88]:
read_one_line(ser)

'dual servo control over serial'

In [89]:
read_all(ser)

''

In [90]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int_byte.to_bytes(1,byteorder='big')
    return out_byte

In [91]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

## Example

In [92]:
#byte1 = 7
#WriteByte(ser,byte1)#<--
#time.sleep(0.1)
#byte2 = 156
#WriteByte(ser,byte2)#<--
#time.sleep(0.1)
#next_line = read_one_line(ser)
#extra = read_all(ser)
#print('next_line: %s' % next_line)
#print('extra: %s' % extra)

# Break an integer into two bytes

In [93]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [94]:
# inputs from user
xll = 5 # x origin
yll = 5 # y origin
w = 30   # width
h = 10   # height
N = 10  # number of steps per side

In [95]:
#define step size
dx = w/N
dy = h/N

#generate bottom coordinants
x_bottom = np.linspace(xll, xll+w-dx, N)  
y_bottom = np.full(N, yll)

#generate right coordinants
x_right = np.full(N, xll + w)  
y_right = np.linspace(yll, yll+h-dy, N)

#generate top coordinants
x_top = np.linspace(xll+w, xll+dx, N)  
y_top = np.full(N,yll+h)

#generate left coordinants
x_left = np.full(N, xll)  
y_left = np.linspace(yll+h, yll, N)

#combine bottom, right, top, left into one array
x_path = np.concatenate((x_bottom, x_right, x_top, x_left), axis=0) 
y_path = np.concatenate((y_bottom, y_right, y_top, y_left), axis=0) 

#combine x and y into one array
tip_path = np.column_stack((x_path, y_path))
tip_path

array([[ 5.        ,  5.        ],
       [ 8.        ,  5.        ],
       [11.        ,  5.        ],
       [14.        ,  5.        ],
       [17.        ,  5.        ],
       [20.        ,  5.        ],
       [23.        ,  5.        ],
       [26.        ,  5.        ],
       [29.        ,  5.        ],
       [32.        ,  5.        ],
       [35.        ,  5.        ],
       [35.        ,  6.        ],
       [35.        ,  7.        ],
       [35.        ,  8.        ],
       [35.        ,  9.        ],
       [35.        , 10.        ],
       [35.        , 11.        ],
       [35.        , 12.        ],
       [35.        , 13.        ],
       [35.        , 14.        ],
       [35.        , 15.        ],
       [32.        , 15.        ],
       [29.        , 15.        ],
       [26.        , 15.        ],
       [23.        , 15.        ],
       [20.        , 15.        ],
       [17.        , 15.        ],
       [14.        , 15.        ],
       [11.        ,

In [96]:
#define link lengths
l1 = 25 # base link
l2 = 22 # tip link

#distance from origin to tip
r_squared = tip_path[:,0]**2 + tip_path[:,1]**2

#law of cos for angle between links
alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
#print(alpha_temp)
alpha = np.arccos(alpha_temp)

#vertical angle theorem for theta 2
theta2 = 180 - alpha*rtd

#triangle in link1 co-ordinant system for psi
psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))

#angle of r to x-axis
beta = np.arctan2(tip_path[:,1],tip_path[:,0])*rtd

#difference in beta and psi is theta 1
theta1 = beta - psi

print("\ntheta 1:\n",theta1)
print("theta 2:\n",theta2)


theta 1:
 [43.99991358 30.94392447 23.36817451 18.59169663 15.35805244 13.04673491
 11.32572307 10.00408369  8.96537067  8.13528465  7.46502399  9.06697857
 10.65463965 12.22585279 13.77861539 15.31108852 16.8216058  18.30867927
 19.77100245 21.20745089 22.61708044 24.44934994 26.61120565 29.17826353
 32.250606   35.9587003  40.4689198  45.98387487 52.72646706 60.88712129
 70.51160663 69.13814026 67.55917392 65.72674757 63.57803544 61.02987089
 57.97128514 54.25386161 49.68087991 43.99991358]
theta 2:
 [164.30752394 158.013597   151.09881666 143.83031895 136.28011062
 128.44902965 120.30118327 111.7716942  102.76243388  93.12677611
  82.63546085  82.05735047  81.37307344  80.58197278  79.68318768
  78.67560555  77.55780415  76.3279819   74.98387325  73.52264528
  71.94076951  82.68797807  92.24031493 100.89973545 108.82793922
 116.10388114 122.74554607 128.71555766 133.91865238 138.19815581
 141.34388643 144.10470475 146.82737279 149.50993702 152.14880072
 154.73801286 157.26812119 15

In [97]:
theta_min = 0
theta_max = 180
min_new = 1000
max_new = 2000

myint = min_new + ((theta1-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print(myint)

myint2 = min_new + ((theta2-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print('\n',myint2)

[1244.44396431 1171.91069151 1129.82319172 1103.2872035  1085.32251358
 1072.48186063 1062.92068374 1055.57824271 1049.80761483 1045.19602583
 1041.47235549 1050.37210315 1059.1924425  1067.92140441 1076.54786326
 1085.06160287 1093.45336555 1101.71488482 1109.8389025  1117.81917162
 1125.65044688 1135.82972187 1147.84003136 1162.10146403 1179.17003335
 1199.77055722 1224.82733225 1255.46597148 1292.92481698 1338.26178496
 1391.73114797 1384.10077925 1375.32874398 1365.14859759 1353.21130798
 1339.05483828 1322.06269525 1301.4103423  1276.00488839 1244.44396431]

 [1912.81957747 1877.85331668 1839.43787031 1799.05732749 1757.11172565
 1713.60572026 1668.33990707 1620.95385665 1570.90241043 1517.37097841
 1459.0858936  1455.87416925 1452.07263023 1447.67762656 1442.68437601
 1437.0866975  1430.87668975 1424.04434386 1416.5770736  1408.45914047
 1399.67094172 1459.37765597 1512.44619403 1560.55408586 1604.59966231
 1645.02156187 1681.91970038 1715.08643146 1743.99251321 1767.76753226
 17

In [98]:
def break_into_two(breakint):
    byts = 0
    MSB = 0
    while byts < breakint-256:
        byts = byts+256
        MSB = MSB+1

    LSB=breakint-byts

    return MSB, LSB

In [99]:
byte1 = np.zeros(len(theta1), dtype=int)
byte2 = np.zeros(len(theta1), dtype=int)
byte3 = np.zeros(len(theta1), dtype=int)
byte4 = np.zeros(len(theta1), dtype=int)

for i in range(len(theta1)):
    byte1[i], byte2[i] = break_into_two(myint[i])
    byte3[i], byte4[i] = break_into_two(myint2[i])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4)

[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5
 5 4 4] 

 [220 147 105  79  61  48  38  31  25  21  17  26  35  43  52  61  69  77
  85  93 101 111 123 138 155 175 200 231  12  58 111 104  95  85  73  59
  42  21 252 220] 


 [7 7 7 7 6 6 6 6 6 5 5 5 5 5 5 5 5 5 5 5 5 5 5 6 6 6 6 6 6 6 6 7 7 7 7 7 7
 7 7 7] 

 [120  85  47   7 221 177 132  84  34 237 179 175 172 167 162 157 150 144
 136 128 119 179 232  24  68 109 145 179 207 231 249   8  23  38  53  67
  81  95 108 120]


In [100]:
# Send all path points to both servos
for i in range(len(theta1)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)

    # read back confirmation from Arduino
    line1 = read_one_line(ser)  # servo 1 bytes echo
    line2 = read_one_line(ser)  # servo 1 int echo
    line3 = read_one_line(ser)  # servo 2 bytes echo
    line4 = read_one_line(ser)  # servo 2 int echo
    print(f"Step {i}: servo1={line2}  servo2={line4}")

    time.sleep(0.5)  # pause between steps so servo has time to move

Step 0: servo1=1244  servo2=1912
Step 1: servo1=1171  servo2=1877
Step 2: servo1=1129  servo2=1839
Step 3: servo1=1103  servo2=1799
Step 4: servo1=1085  servo2=1757
Step 5: servo1=1072  servo2=1713
Step 6: servo1=1062  servo2=1668
Step 7: servo1=1055  servo2=1620
Step 8: servo1=1049  servo2=1570
Step 9: servo1=1045  servo2=1517
Step 10: servo1=1041  servo2=1459
Step 11: servo1=1050  servo2=1455
Step 12: servo1=1059  servo2=1452
Step 13: servo1=1067  servo2=1447
Step 14: servo1=1076  servo2=1442
Step 15: servo1=1085  servo2=1437
Step 16: servo1=1093  servo2=1430
Step 17: servo1=1101  servo2=1424
Step 18: servo1=1109  servo2=1416
Step 19: servo1=1117  servo2=1408
Step 20: servo1=1125  servo2=1399
Step 21: servo1=1135  servo2=1459
Step 22: servo1=1147  servo2=1512
Step 23: servo1=1162  servo2=1560
Step 24: servo1=1179  servo2=1604
Step 25: servo1=1199  servo2=1645
Step 26: servo1=1224  servo2=1681
Step 27: servo1=1255  servo2=1715
Step 28: servo1=1292  servo2=1743
Step 29: servo1=1338  se

In [66]:
#byte3, byte4 = break_into_two(1200)

In [67]:
#WriteByte(ser, MSB)
#WriteByte(ser, LSB)

In [69]:

WriteByte(ser,byte1)#<--
time.sleep(0.1)

WriteByte(ser,byte2)#<--
time.sleep(0.1)

#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

AttributeError: 'numpy.ndarray' object has no attribute 'to_bytes'

In [ ]:
byte3, byte4 = break_into_two(1500)

In [ ]:
WriteByte(ser,byte3)#<--
time.sleep(0.1)

WriteByte(ser,byte4)#<--
time.sleep(0.1)
#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

In [ ]:
ser.close()

In [ ]:
for i in range(1100, 1900, 50):
    byte3, byte4 = break_into_two(i)
    WriteByte(ser,byte3)#<--
    time.sleep(0.1)

    WriteByte(ser,byte4)#<--
    time.sleep(0.1)
    #WriteByte(ser,byte3)#<--
    #time.sleep(0.1)

    #WriteByte(ser,byte4)#<--
    #time.sleep(0.1)
    next_line = read_one_line(ser)
    extra = read_all(ser)
    print('next_line: %s' % next_line)
    print('extra: %s' % extra)
    time.sleep(0.5)
    print(i) 

- How do we break this into two bytes?
- How do we find the most significant byte?
- How do we find the least significant byte?

In [ ]:
ser.close()